In [1]:
from moabb.paradigms import LeftRightImagery, MotorImagery
from moabb.datasets import *

sfreq=250
paradigm = LeftRightImagery(resample=sfreq)

datasets = [
    BNCI2014_004()
]


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from sklearn.model_selection import StratifiedKFold
from hoda.hoda import BTTDA, GreedyBTTDA, HODA, trunc_eigh
from hoda.cov import mode_scatter, ledoit_wolf_shrinkage
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from mne.decoding import Scaler
import  warnings
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectFwe
from sklearn.preprocessing import StandardScaler, FunctionTransformer
import warnings
from joblib import parallel_backend
from joblib import Parallel
from hoda.classification import SelectF
from sklearn.pipeline import make_pipeline
import tensorly as tl

clf = make_pipeline(
    SelectF(alpha=.05),
    FunctionTransformer(tl.to_numpy),
    StandardScaler(),
    LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')
)

bttda = GreedyBTTDA(
    max_blocks=10,
    truncate=False,
    hoda_params=dict(
        rank=None,
        max_iter=128,
        tol=1e-8,
        init ='random',
        shrinkage='lw',
        toeplitz=None,
        obj='tr',
        solver='lanczos',
        taper=False,
        extra_train_info=False,
        verbose=False,
        random_state=42,
        delta=None,
       
    ),
    verbose=True,
    extra_train_info=True,
    cv=StratifiedKFold(random_state=42, shuffle=True),
    clf=clf,
    scoring='accuracy',
)

In [ ]:
import tensorly as tl
import pandas as pd
from hoda.tensorize import stf_tensor

select_infos = []
train_infos = []
for dataset in datasets:
    for subject in dataset.subject_list:
        epochs, labels, meta = paradigm.get_data(
            dataset=dataset, 
             subjects=[subject],
             return_epochs=True
        )
        for session in meta['session'].unique():
            idc = meta['session'] == session
            epochs_ses = epochs[idc]
            labels_ses = labels[idc]
            meta_ses = meta[idc]

            X = epochs_ses.get_data()
            X = stf_tensor(X, sfreq=epochs.info['sfreq'], normalize=True,log=True, n_freqs=4, bin_freq=4)
            X = tl.tensor(X)
            y = labels_ses

            print(f'dataset={dataset.code} subject={subject} session={session}')
            bttda.fit(X,y, test=True)
            subj_select_info = pd.DataFrame(bttda.model_select_info_best_) 
            subj_select_info['dataset'] = dataset.code
            subj_select_info['subject'] = subject
            subj_select_info['session'] = session
            subj_train_info = pd.DataFrame(bttda.train_info_)
            subj_train_info['dataset'] = dataset.code
            subj_train_info['subject'] = subject
            subj_train_info['session'] = session
            select_infos.append(subj_select_info.reset_index())
            train_infos.append(subj_train_info.reset_index())
            # save
            select_info = pd.concat(select_infos, ignore_index=True)
            train_info = pd.concat(train_infos, ignore_index=True)
            select_info.to_csv('block_lr_select.csv')
            train_info.to_csv('block_lr_train.csv')


Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns
720 matching events found
No baseline correction applied


/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 120 events (all good), 3 – 7.5 s (baseline off), ~3.1 MB, data loaded,
 'left_hand': 60
 'right_hand': 60>
  warn(f"warnEpochs {epochs}")
/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 120 events (all good), 3 – 7.5 s (baseline off), ~3.1 MB, data loaded,
 'left_hand': 60
 'right_hand': 60>
  warn(f"warnEpochs {epochs}")
/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 160 events (all good), 3 – 7.5 s (baseline off), ~4.1 MB, data loaded,
 'left_hand': 80
 'right_hand': 80>
  warn(f"warnEpochs {epochs}")
/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: war

dataset=BNCI2014-004 subject=1 session=0train
Model selection block 1/10...

Trying rank (1, 1, 1)	score: 0.7811
Trying rank (2, 2, 2)	score: 0.8542
Trying rank (3, 4, 4)	score: 0.8426
Trying rank (3, 4, 8)	score: 0.8016
Trying rank (3, 4, 16)	score: 0.8542
Trying rank (3, 4, 18)	score: 0.8537

Selected rank (2, 2, 2) with score 0.8542
New ranks: [(2, 2, 2)]


Model selection block 2/10...

Trying rank (1, 1, 1)	score: 0.8021
Trying rank (2, 2, 2)	score: 0.8116
Trying rank (3, 4, 4)	score: 0.8542
Trying rank (3, 4, 8)	score: 0.7495
Trying rank (3, 4, 16)	score: 0.8026
Trying rank (3, 4, 18)	score: 0.8232

Selected rank (3, 4, 4) with score 0.8542
New ranks: [(2, 2, 2), (3, 4, 4)]


Model selection block 3/10...

Trying rank (1, 1, 1)	score: 0.8642
Trying rank (2, 2, 2)	score: 0.8537
Trying rank (3, 4, 4)	score: 0.7900
Trying rank (3, 4, 8)	score: 0.8011
Trying rank (3, 4, 16)	score: 0.8005
Trying rank (3, 4, 18)	score: 0.8011

Selected rank (1, 1, 1) with score 0.8642
New ranks: [(2, 2

/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 120 events (all good), 3 – 7.5 s (baseline off), ~3.1 MB, data loaded,
 'left_hand': 60
 'right_hand': 60>
  warn(f"warnEpochs {epochs}")
/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 120 events (all good), 3 – 7.5 s (baseline off), ~3.1 MB, data loaded,
 'left_hand': 60
 'right_hand': 60>
  warn(f"warnEpochs {epochs}")
/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 160 events (all good), 3 – 7.5 s (baseline off), ~4.1 MB, data loaded,
 'left_hand': 80
 'right_hand': 80>
  warn(f"warnEpochs {epochs}")
/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: war

dataset=BNCI2014-004 subject=2 session=0train
Model selection block 1/10...

Trying rank (1, 1, 1)	score: 0.6779
Trying rank (2, 2, 2)	score: 0.6042
Trying rank (3, 4, 4)	score: 0.5011
Trying rank (3, 4, 8)	score: 0.5632
Trying rank (3, 4, 16)	score: 0.5737
Trying rank (3, 4, 18)	score: 0.6453

Selected rank (1, 1, 1) with score 0.6779
New ranks: [(1, 1, 1)]


Model selection block 2/10...

Trying rank (1, 1, 1)	score: 0.5842
Trying rank (2, 2, 2)	score: 0.5642
Trying rank (3, 4, 4)	score: 0.5626
Trying rank (3, 4, 8)	score: 0.6042
Trying rank (3, 4, 16)	score: 0.6047
Trying rank (3, 4, 18)	score: 0.6047

Selected rank (3, 4, 16) with score 0.6047
New ranks: [(1, 1, 1), (3, 4, 16)]


Model selection block 3/10...

Trying rank (1, 1, 1)	score: 0.5532
Trying rank (2, 2, 2)	score: 0.5632
Trying rank (3, 4, 4)	score: 0.5947
Trying rank (3, 4, 8)	score: 0.5737
Trying rank (3, 4, 16)	score: 0.5847
Trying rank (3, 4, 18)	score: 0.6153

Selected rank (3, 4, 18) with score 0.6153
New ranks: [(1

...

dataset=BNCI2014-004 subject=3 session=0train
Model selection block 1/10...

Trying rank (1, 1, 1)	score: 0.4379
Trying rank (2, 2, 2)	score: 0.5316
Trying rank (3, 4, 4)	score: 0.5216
Trying rank (3, 4, 8)	score: 0.5005
Trying rank (3, 4, 16)	score: 0.5105
Trying rank (3, 4, 18)	score: 0.4584

Selected rank (2, 2, 2) with score 0.5316
New ranks: [(2, 2, 2)]


Model selection block 2/10...

Trying rank (1, 1, 1)	score: 0.5005
Trying rank (2, 2, 2)	score: 0.5321
Trying rank (3, 4, 4)	score: 0.5426
Trying rank (3, 4, 8)	score: 0.5005
Trying rank (3, 4, 16)	score: 0.5226
Trying rank (3, 4, 18)	score: 0.4800

Selected rank (3, 4, 4) with score 0.5426
New ranks: [(2, 2, 2), (3, 4, 4)]


Model selection block 3/10...

Trying rank (1, 1, 1)	score: 0.5332
Trying rank (2, 2, 2)	score: 0.5011
Trying rank (3, 4, 4)	score: 0.5116
Trying rank (3, 4, 8)	score: 0.5326
Trying rank (3, 4, 16)	score: 0.5326
Trying rank (3, 4, 18)	score: 0.4905

Selected rank (1, 1, 1) with score 0.5332
New ranks: [(2, 2

/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 120 events (all good), 3 – 7.5 s (baseline off), ~3.1 MB, data loaded,
 'left_hand': 60
 'right_hand': 60>
  warn(f"warnEpochs {epochs}")
/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 140 events (all good), 3 – 7.5 s (baseline off), ~3.6 MB, data loaded,
 'left_hand': 70
 'right_hand': 70>
  warn(f"warnEpochs {epochs}")
/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 160 events (all good), 3 – 7.5 s (baseline off), ~4.1 MB, data loaded,
 'left_hand': 80
 'right_hand': 80>
  warn(f"warnEpochs {epochs}")
/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: war

dataset=BNCI2014-004 subject=4 session=0train
Model selection block 1/10...

Trying rank (1, 1, 1)	score: 0.9268
Trying rank (2, 2, 2)	score: 0.8853
Trying rank (3, 4, 4)	score: 0.8847
Trying rank (3, 4, 8)	score: 0.8753
Trying rank (3, 4, 16)	score: 0.8553
Trying rank (3, 4, 18)	score: 0.8868

Selected rank (1, 1, 1) with score 0.9268
New ranks: [(1, 1, 1)]


Model selection block 2/10...

Trying rank (1, 1, 1)	score: 0.8753
Trying rank (2, 2, 2)	score: 0.8858
Trying rank (3, 4, 4)	score: 0.8437
Trying rank (3, 4, 8)	score: 0.8542
Trying rank (3, 4, 16)	score: 0.8653
Trying rank (3, 4, 18)	score: 0.8963

Selected rank (3, 4, 18) with score 0.8963
New ranks: [(1, 1, 1), (3, 4, 18)]


Model selection block 3/10...

Trying rank (1, 1, 1)	score: 0.8853
Trying rank (2, 2, 2)	score: 0.8753
Trying rank (3, 4, 4)	score: 0.8853
Trying rank (3, 4, 8)	score: 0.8558
Trying rank (3, 4, 16)	score: 0.8542
Trying rank (3, 4, 18)	score: 0.8963

Selected rank (3, 4, 18) with score 0.8963
New ranks: [(1

/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 120 events (all good), 3 – 7.5 s (baseline off), ~3.1 MB, data loaded,
 'left_hand': 60
 'right_hand': 60>
  warn(f"warnEpochs {epochs}")
/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 140 events (all good), 3 – 7.5 s (baseline off), ~3.6 MB, data loaded,
 'left_hand': 70
 'right_hand': 70>
  warn(f"warnEpochs {epochs}")
/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 160 events (all good), 3 – 7.5 s (baseline off), ~4.1 MB, data loaded,
 'left_hand': 80
 'right_hand': 80>
  warn(f"warnEpochs {epochs}")
/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: war

dataset=BNCI2014-004 subject=5 session=0train
Model selection block 1/10...

Trying rank (1, 1, 1)	score: 0.7489
Trying rank (2, 2, 2)	score: 0.7700
Trying rank (3, 4, 4)	score: 0.6758
Trying rank (3, 4, 8)	score: 0.6868
Trying rank (3, 4, 16)	score: 0.7495
Trying rank (3, 4, 18)	score: 0.7289

Selected rank (2, 2, 2) with score 0.7700
New ranks: [(2, 2, 2)]


Model selection block 2/10...

Trying rank (1, 1, 1)	score: 0.6347
Trying rank (2, 2, 2)	score: 0.6447
Trying rank (3, 4, 4)	score: 0.6974
Trying rank (3, 4, 8)	score: 0.7184
Trying rank (3, 4, 16)	score: 0.7189
Trying rank (3, 4, 18)	score: 0.7179

Selected rank (3, 4, 16) with score 0.7189
New ranks: [(2, 2, 2), (3, 4, 16)]


Model selection block 3/10...

Trying rank (1, 1, 1)	score: 0.7289
Trying rank (2, 2, 2)	score: 0.7389
Trying rank (3, 4, 4)	score: 0.6874
Trying rank (3, 4, 8)	score: 0.7189
Trying rank (3, 4, 16)	score: 0.6663
Trying rank (3, 4, 18)	score: 0.6774

Selected rank (2, 2, 2) with score 0.7389
New ranks: [(2,

In [ ]:
select_info

In [ ]:
select_info

In [ ]:
import seaborn as sns

idx = ['dataset', 'subject', 'session', 'block']
df = select_info.groupby(idx)[['train_score', 'val_score', 'test_score']].aggregate('mean')
df = df.melt(var_name='split', ignore_index=False)
df = df.reset_index()
sns.lineplot(data=df, x='block', y='value', hue='dataset', style='split',errorbar=None)

In [ ]:
train_info.to_csv('block_mi_train.csv')
train_info

In [ ]:
df = train_info.groupby(idx)
df = df['nmse'].aggregate('mean')
df = df.reset_index()
sns.lineplot(data=df, x='block',y='nmse', hue='dataset')